# Automated ICD-9 Coding from Clinical Notes
### A self-learning tutorial: from a bag-of-words baseline to a CNN with per-label attention

When a patient leaves the hospital, a clinician writes a free-text **discharge summary**, and a trained
medical coder reads it and assigns **ICD diagnosis codes**: a standardized label for every condition. Those
codes drive billing, quality reporting, and research. Coding by hand is slow and inconsistent, which makes it
a natural target for machine learning.

**In this tutorial we teach a model to do that job:** read a discharge summary and predict its diagnosis codes.
We build the idea up in two stages so it is easy to follow and easy to rebuild:

1. A simple, transparent **TF-IDF + logistic regression** baseline.
2. A **convolutional neural network with per-label attention** (the CAML architecture, Mullenbach et al. 2018),
   a genuine deep model that not only predicts codes but points to the exact phrase in the note behind each one.

**Data.** MIMIC-III, a public but *credentialed* critical-care database, used under its Data Use Agreement.
We use three tables: `NOTEEVENTS` (the notes), `DIAGNOSES_ICD` (the codes), and `D_ICD_DIAGNOSES` (code titles).
No data is shipped with this notebook; place your own copies in `data/` (see the README).

**Runtime.** The notebook runs end to end on a standard CPU in about 20 minutes.

**Roadmap.** Problem framing -> cohort -> preprocessing -> metrics -> baseline -> CNN + attention -> results -> explanation -> limitations.

> **Before you run:** place three MIMIC-III CSV files in a `data/` folder next to this notebook: `NOTEEVENTS.csv`, `DIAGNOSES_ICD.csv`, and `D_ICD_DIAGNOSES.csv`. MIMIC-III is credentialed; download it from PhysioNet and add the files yourself. No data is included with this notebook.

## 0. Setup and configuration

We fix random seeds for reproducibility and gather every knob in one place so the whole tutorial is easy to
re-run and to tweak.

In [ ]:
%matplotlib inline
import os, re, json, time, random, copy
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- reproducibility: seed every source of randomness we use ---
SEED = 13
random.seed(SEED); np.random.seed(SEED)

# --- paths: the three MIMIC-III CSVs go in data/ (see README) ---
ROOT  = os.getcwd()
DATA  = os.path.join(ROOT, "data")
CACHE = os.path.join(ROOT, "cache")       # processed cohort + trained weights (git-ignored)
ART   = os.path.join(ROOT, "artifacts")   # small json metrics + label list
RES   = os.path.join(ROOT, "results")     # figures + the deck
for d in (CACHE, ART, RES): os.makedirs(d, exist_ok=True)

# --- problem configuration ---
N_LABELS = 50            # use the 50 most frequent ICD-9 codes: the standard MIMIC-III benchmark
SAMPLE_FRACTION = 0.33   # subsample patients to benchmark scale so CPU training is quick; 1.0 = full cohort

# palette, so the notebook figures match the slides
NAVY, TEAL, SLATE, LIGHT, ACCENT = "#162947", "#1F7A8C", "#333A45", "#F2F4F7", "#C0392B"
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans",
                     "axes.edgecolor": "#B9C0CC", "axes.linewidth": 0.8})
print("Ready. Working directory:", ROOT)

## 1. The clinical problem and the data

A human coder turns one document (the note) into a set of labels (the codes). We will learn that mapping. First
we assemble the cohort from three tables:

- **`NOTEEVENTS`** holds every clinical note. We keep only the *discharge summaries*.
- **`DIAGNOSES_ICD`** lists the ICD-9 codes assigned to each admission. These are our labels.
- **`D_ICD_DIAGNOSES`** gives each code a readable title, for figures and explanations.

### 1.1 Build the cohort

Three decisions in the cell below are worth calling out, because they are where tutorials usually go wrong:

- **One document per admission.** A discharge summary can arrive as a main report plus later addenda. We
  concatenate them so each admission is a single document.
- **Top-50 labels.** Real coding spans thousands of codes. We keep the 50 most frequent, a top-50 MIMIC-III benchmark-style task, so the problem is tractable and teachable, and keep admissions that carry at least one of them.
- **Text cleaning.** We strip MIMIC's `[**de-identification**]` placeholders, lowercase, and keep alphabetic
  word tokens. A rule you can state in one sentence beats a black box.

The build is cache-aware: it scans the 3.8 GB notes file once, then reloads instantly on later runs.

In [ ]:
COHORT_PKL = os.path.join(CACHE, "cohort.pkl")

def build_cohort():
    """Return (cohort dataframe, list of the 50 ICD-9 code columns)."""
    # 1) discharge summaries from NOTEEVENTS, read in 100k-row chunks (the file is ~3.8 GB)
    print("Scanning NOTEEVENTS for discharge summaries ...")
    cols = ["SUBJECT_ID", "HADM_ID", "CATEGORY", "DESCRIPTION", "ISERROR", "TEXT"]
    parts = []
    for ch in pd.read_csv(os.path.join(DATA, "NOTEEVENTS.csv"), usecols=cols, chunksize=100000,
                          dtype={"SUBJECT_ID": "Int64", "HADM_ID": "Int64", "ISERROR": "float"}):
        keep = ch[(ch.CATEGORY == "Discharge summary") & (ch.ISERROR.isna())].dropna(subset=["HADM_ID"])
        if len(keep):
            parts.append(keep[["SUBJECT_ID", "HADM_ID", "DESCRIPTION", "TEXT"]])
    notes = pd.concat(parts, ignore_index=True)

    # one document per admission (report + any addenda)
    notes = notes.sort_values(["HADM_ID", "DESCRIPTION"])
    adm = notes.groupby("HADM_ID").agg(SUBJECT_ID=("SUBJECT_ID", "first"),
                                       TEXT=("TEXT", lambda s: "\n".join(s))).reset_index()

    # 2) diagnoses -> the set of ICD-9 codes for each admission
    dx = pd.read_csv(os.path.join(DATA, "DIAGNOSES_ICD.csv"), dtype=str).dropna(subset=["HADM_ID", "ICD9_CODE"])
    dx["HADM_ID"] = dx["HADM_ID"].astype(int)
    codes_per_adm = dx.groupby("HADM_ID")["ICD9_CODE"].apply(set)

    # the 50 most frequent codes among admissions that have a discharge summary
    dx_note = dx[dx.HADM_ID.isin(set(adm.HADM_ID))]
    top_codes = (dx_note.groupby("ICD9_CODE")["HADM_ID"].nunique()
                 .sort_values(ascending=False).head(N_LABELS).index.tolist())

    # 3) keep admissions with >=1 top code, then make one 0/1 column per code
    top_set = set(top_codes)
    adm["labels"] = adm.HADM_ID.map(lambda h: codes_per_adm.get(h, set()) & top_set)
    adm = adm[adm.labels.map(len) > 0].reset_index(drop=True)
    for c in top_codes:
        adm[c] = adm.labels.map(lambda s: int(c in s))

    # 4) clean text: remove [**de-id**], lowercase, keep alphabetic tokens
    deid, tok = re.compile(r"\[\*\*.*?\*\*\]"), re.compile(r"[a-z]+")
    adm["text"] = adm.TEXT.map(lambda t: " ".join(tok.findall(deid.sub(" ", t.lower()))))
    adm["n_tokens"] = adm.text.map(lambda t: len(t.split()))

    # 5) split by PATIENT, never by note, so no patient appears in two splits (prevents leakage)
    rng = np.random.default_rng(SEED)
    patients = np.asarray(adm.SUBJECT_ID.unique()); rng.shuffle(patients)
    n, = patients.shape; n_tr, n_va = int(0.8 * n), int(0.1 * n)
    split = {p: ("train" if i < n_tr else "val" if i < n_tr + n_va else "test")
             for i, p in enumerate(patients)}
    adm["split"] = adm.SUBJECT_ID.map(split)

    # 6) subsample patients to benchmark scale to keep CPU training quick
    if SAMPLE_FRACTION < 1.0:
        rng2 = np.random.default_rng(7)
        keep = set(rng2.choice(adm.SUBJECT_ID.unique(),
                               size=int(SAMPLE_FRACTION * adm.SUBJECT_ID.nunique()), replace=False))
        adm = adm[adm.SUBJECT_ID.isin(keep)].reset_index(drop=True)

    return adm[["HADM_ID", "SUBJECT_ID", "split", "text", "n_tokens"] + top_codes], top_codes

if os.path.exists(COHORT_PKL):
    cohort = pd.read_pickle(COHORT_PKL)
    CODES = [c for c in cohort.columns if c not in ("HADM_ID", "SUBJECT_ID", "split", "text", "n_tokens")]
    print("Loaded cached cohort:", cohort.shape)
else:
    t0 = time.time()
    cohort, CODES = build_cohort()
    cohort.to_pickle(COHORT_PKL)
    print("Built cohort in %.0fs: %s" % (time.time() - t0, cohort.shape))

Attach readable titles to each code (from `D_ICD_DIAGNOSES`) and save a small metadata file the slide deck reuses.

In [ ]:
dxd = pd.read_csv(os.path.join(DATA, "D_ICD_DIAGNOSES.csv"), dtype=str)
SHORT = dict(zip(dxd.ICD9_CODE, dxd.SHORT_TITLE))
LONG  = dict(zip(dxd.ICD9_CODE, dxd.LONG_TITLE))
TITLE = {c: SHORT.get(c, c) for c in CODES}

# label metadata (code, title, how many cohort admissions carry it), ordered by frequency
LAB = [{"code": c, "short": SHORT.get(c, c), "long": LONG.get(c, c), "n_adm": int(cohort[c].sum())} for c in CODES]
json.dump(LAB, open(os.path.join(ART, "labels.json"), "w"), indent=2)

for l in LAB[:8]:
    print(f"{l['code']:>6}  {l['n_adm']:>5}  {l['short']}")

### 1.2 A first look at the cohort

How big is it, how many codes does a note carry, and how long are these notes? The last question matters because
the neural network will have to cap the length it reads.

In [ ]:
tr = cohort[cohort.split == "train"]; va = cohort[cohort.split == "val"]; te = cohort[cohort.split == "test"]
stats = {"n_admissions": int(len(cohort)), "n_patients": int(cohort.SUBJECT_ID.nunique()), "n_labels": N_LABELS,
         "avg_labels_per_note": round(float(cohort[CODES].sum(1).mean()), 2),
         "median_tokens": int(cohort.n_tokens.median()), "p90_tokens": int(cohort.n_tokens.quantile(0.9)),
         "split_counts": {s: int((cohort.split == s).sum()) for s in ("train", "val", "test")}}
json.dump(stats, open(os.path.join(ART, "cohort_stats.json"), "w"), indent=2)
print(json.dumps(stats, indent=2))

# a peek at one cleaned note (first 40 tokens)
print("\nExample cleaned note (first 40 tokens):\n", " ".join(cohort.text.iloc[0].split()[:40]), "...")

In [ ]:
def style(ax):
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    ax.tick_params(colors=SLATE, labelsize=9); ax.set_axisbelow(True)

# label frequency (top 15) and note-length distribution
top = LAB[:15][::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([f"{l['short'][:26]} ({l['code']})" for l in top], [l["n_adm"] for l in top], color=TEAL)
for i, l in enumerate(top): ax.text(l["n_adm"], i, f" {l['n_adm']:,}", va="center", fontsize=8, color=SLATE)
ax.grid(axis="x", color="#E6E9EF", lw=0.8); style(ax)
ax.set_title("Most frequent ICD-9 codes in the cohort", color=NAVY, fontsize=12, fontweight="bold", loc="left")
fig.tight_layout(); fig.savefig(os.path.join(RES, "fig_labels.png"), facecolor="white", bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(cohort.n_tokens.clip(upper=4000), bins=40, color=TEAL, alpha=0.85)
ax.grid(axis="y", color="#E6E9EF", lw=0.8); style(ax)
ax.set_xlabel("Tokens per discharge summary", color=SLATE); ax.set_ylabel("Admissions", color=SLATE)
ax.set_title("Discharge summaries are long documents", color=NAVY, fontsize=12, fontweight="bold", loc="left")
fig.tight_layout(); fig.savefig(os.path.join(RES, "fig_note_len.png"), facecolor="white", bbox_inches="tight"); plt.show()

## 2. How we will measure success

Plain accuracy is a trap here: most of the 50 codes are absent from any given note, so a model that says "no"
to everything scores high and helps no one. We use three metrics instead:

- **Micro-F1** pools every label decision together. It is dominated by the common codes and reports overall how
  many code assignments were right.
- **Macro-F1** averages F1 across the 50 codes equally, so a rare code counts as much as hypertension. It exposes
  whether the model only learned the easy, frequent codes.
- **Precision@k** asks: of the model's *k* most confident codes, how many are correct. That mirrors real use,
  where the tool hands a coder a short ranked list to confirm.

We choose the decision threshold on the **validation** set and report on a **test** set we never touch during
development, so the numbers are not inflated by tuning.

In [ ]:
from sklearn.metrics import f1_score

def precision_at_k(P, Y, k):
    """Average over notes of (correct codes among the model's top-k) / k."""
    idx = np.argsort(-P, axis=1)[:, :k]
    return float((np.take_along_axis(Y, idx, axis=1).sum(1) / k).mean())

def best_threshold(P, Y):
    """Pick the probability cutoff that maximizes micro-F1 on given (usually validation) data."""
    return max(np.linspace(0.05, 0.9, 18),
               key=lambda t: f1_score(Y, (P >= t).astype(int), average="micro", zero_division=0))

def evaluate(P, Y, thr):
    pred = (P >= thr).astype(int)
    return {"micro_f1": round(f1_score(Y, pred, average="micro", zero_division=0), 4),
            "macro_f1": round(f1_score(Y, pred, average="macro", zero_division=0), 4),
            "p@5": round(precision_at_k(P, Y, 5), 4), "p@8": round(precision_at_k(P, Y, 8), 4)}

# label matrices (same column order as CODES) for each split
Ytr = tr[CODES].values.astype(int); Yva = va[CODES].values.astype(int); Yte = te[CODES].values.astype(int)
print("label matrices:", Ytr.shape, Yva.shape, Yte.shape)

## 3. A strong, simple baseline: TF-IDF + logistic regression

Represent each note by **TF-IDF**: count how often each word appears, down-weighted if it is common everywhere,
so distinctive terms like *dialysis* outweigh *the*. Then train **one logistic regression per code** (one-vs-rest):
50 independent yes/no classifiers.

This is deliberately old-fashioned. It is fast, transparent, and genuinely hard to beat on frequent codes, which
is exactly why you build it first: it is the bar the neural network has to clear.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

t0 = time.time()
vec = TfidfVectorizer(min_df=3, max_features=40000, ngram_range=(1, 2), sublinear_tf=True)
Xtr = vec.fit_transform(tr.text); Xva = vec.transform(va.text); Xte = vec.transform(te.text)
for X in (Xtr, Xva, Xte): X.sort_indices()   # canonical CSR (avoids an in-place sort during fit)
print("TF-IDF matrix:", Xtr.shape, "in %.0fs" % (time.time() - t0))

# choose the regularization strength C on the validation set
best = None
for C in [4, 16]:
    clf = OneVsRestClassifier(LogisticRegression(C=C, max_iter=400, solver="liblinear"), n_jobs=1).fit(Xtr, Ytr)
    Pva = clf.predict_proba(Xva); thr = best_threshold(Pva, Yva); m = evaluate(Pva, Yva, thr)["micro_f1"]
    print(f"  C={C}: validation micro-F1 = {m}")
    if best is None or m > best[0]:
        best = (m, C, clf, thr)
_, C_base, base_clf, base_thr = best

base_res = {"model": "tfidf_logreg", "C": C_base, "threshold": round(float(base_thr), 3),
            "val": evaluate(base_clf.predict_proba(Xva), Yva, base_thr),
            "test": evaluate(base_clf.predict_proba(Xte), Yte, base_thr),
            "train_seconds": round(time.time() - t0, 1)}
json.dump(base_res, open(os.path.join(ART, "baseline_metrics.json"), "w"), indent=2)
print("\nBaseline test metrics:", base_res["test"])

The baseline lands around **micro-F1 0.60** and **macro-F1 0.53** on the test set in about five minutes of compute (the runtime above includes trying two values of `C`). That is a real, respectable result. Now we ask
whether a deep model can do better, and do something the baseline cannot.

## 4. The deep model: a CNN that reads with per-label attention

The architecture, left to right:

`tokens -> word embeddings -> 1-D convolution -> per-label attention -> 50 code probabilities`

- **Embeddings** map each word to a learned vector, so the model can discover that *renal* and *kidney* are related.
- **A 1-D convolution** slides a small window (here 10 words) across the note. It is a learned *phrase detector*
  that fires on patterns like *acute renal failure* wherever they occur.
- **Per-label attention** is the key idea: each of the 50 codes keeps *its own* attention over the note and
  focuses on the phrases relevant to it. Heart failure looks at different words than diabetes.
- Each code's focused summary passes through a **sigmoid** to give its probability.

This is the **CAML** architecture (Mullenbach et al., 2018). It is a real convolutional network, yet it trains
from scratch on a CPU in about 16 minutes, and because attention is explicit the phrases it attended to for each code are available directly (Section 6).

### 4.1 From words to numbers

We build a vocabulary from the training split only (never peeking at validation or test), map each note to a
list of word ids, and cap the length we read.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(SEED); torch.set_num_threads(os.cpu_count() or 4)

MAXLEN, MIN_DF, EMB, NF, KERNEL, BATCH, LR, EPOCHS, PATIENCE, DROP = 1500, 3, 100, 128, 10, 32, 1e-3, 15, 3, 0.2

# vocabulary from the TRAIN split only; index 0 = padding, 1 = unknown word
counter = Counter()
for t in tr.text: counter.update(t.split())
itos = ["<pad>", "<unk>"] + [w for w, c in counter.most_common() if c >= MIN_DF]
stoi = {w: i for i, w in enumerate(itos)}
V = len(itos); print("vocabulary size:", V)

def encode(text):
    ids = [stoi.get(w, 1) for w in text.split()[:MAXLEN]]   # truncate to the first MAXLEN words
    return ids if ids else [1]

def prep(dframe):
    return [encode(t) for t in dframe.text], dframe[CODES].values.astype(np.float32)

Xtr_ids, Ytr_f = prep(tr); Xva_ids, Yva_f = prep(va); Xte_ids, Yte_f = prep(te)

def pad(batch):
    """Pad a list of id-lists to the longest in the batch -> LongTensor (B, T)."""
    m = max(len(x) for x in batch)
    arr = np.zeros((len(batch), m), dtype=np.int64)
    for i, x in enumerate(batch): arr[i, :len(x)] = x
    return torch.from_numpy(arr)

### 4.2 The model

The whole network is about 20 lines. The comments trace the shape of the data at every step, because keeping the
dimensions straight is most of the battle with attention.

In [ ]:
class CAML(nn.Module):
    """Convolution + one attention distribution per label (Mullenbach et al., 2018)."""
    def __init__(self, V, E, F, K, C):
        super().__init__()
        self.emb  = nn.Embedding(V, E, padding_idx=0)     # word -> E-dim vector
        self.drop = nn.Dropout(DROP)
        self.conv = nn.Conv1d(E, F, K, padding=K // 2)    # phrase detector: E channels -> F filters
        self.U    = nn.Linear(F, C)                       # one attention query vector per label
        self.final = nn.Linear(F, C)                      # one output weight vector per label
        for w in (self.conv.weight, self.U.weight, self.final.weight): nn.init.xavier_uniform_(w)

    def forward(self, x):                                 # x: (B, T) word ids
        mask = (x == 0)                                   # (B, T) True where padding
        h = self.drop(self.emb(x)).transpose(1, 2)        # (B, E, T)
        h = torch.tanh(self.conv(h))[:, :, :x.size(1)]    # (B, F, T) conv, cropped back to input length
        h = h.transpose(1, 2)                             # (B, T, F)
        scores = self.U(h).masked_fill(mask.unsqueeze(-1), -1e9)  # (B, T, C) attention scores, pad = -inf
        a = torch.softmax(scores, dim=1)                  # (B, T, C) softmax over tokens, once per label
        ctx = torch.einsum("btf,btc->bcf", h, a)          # (B, C, F) label-specific document vectors
        logits = (ctx * self.final.weight).sum(-1) + self.final.bias  # (B, C) one score per label
        return logits, a

model = CAML(V, EMB, NF, KERNEL, len(CODES))
print(model)
print("trainable parameters:", sum(p.numel() for p in model.parameters()))

### 4.3 Training

We minimize binary cross-entropy over all 50 labels at once, check validation micro-F1 after each epoch, and keep
the best epoch (early stopping). A small trick keeps it fast on CPU: we sort notes by length so each batch pads to
a similar size and wastes little compute on padding.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=LR)
crit = nn.BCEWithLogitsLoss()

def predict(model, X):
    """Sigmoid probabilities for a list of encoded notes, in order -> (N, 50)."""
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), 64):
            logits, _ = model(pad(X[i:i + 64])); out.append(torch.sigmoid(logits).numpy())
    return np.vstack(out)

# length-bucketed batches: less padding, faster training
order = np.argsort([len(x) for x in Xtr_ids])
batches = [order[i:i + BATCH] for i in range(0, len(order), BATCH)]
rng = np.random.default_rng(SEED)

t0 = time.time(); best_f1, best_state, best_thr, curve, no_improve = -1, None, 0.5, [], 0
for ep in range(EPOCHS):
    model.train(); rng.shuffle(batches); running = 0.0
    for b in batches:
        xb = pad([Xtr_ids[i] for i in b]); yb = torch.from_numpy(Ytr_f[b])
        opt.zero_grad(); logits, _ = model(xb); loss = crit(logits, yb)
        loss.backward(); opt.step(); running += loss.item() * len(b)
    Pva = predict(model, Xva_ids); thr = best_threshold(Pva, Yva); m = evaluate(Pva, Yva, thr)
    curve.append(m["micro_f1"])
    print(f"epoch {ep+1:2d}  loss={running/len(Xtr_ids):.4f}  val micro-F1={m['micro_f1']}  macro-F1={m['macro_f1']}  ({time.time()-t0:.0f}s)")
    if m["micro_f1"] > best_f1:
        best_f1, best_state, best_thr, no_improve = m["micro_f1"], copy.deepcopy(model.state_dict()), thr, 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("early stop"); break

model.load_state_dict(best_state)   # roll back to the best epoch

### 4.4 How did it do?

We evaluate the best epoch on the untouched test set with the same metrics as the baseline, and save the model,
vocabulary, and metrics for reuse by the explanation section and the slide deck.

In [ ]:
cnn_res = {"model": "cnn_attention", "threshold": round(float(best_thr), 3), "epochs_run": len(curve),
           "train_seconds": round(time.time() - t0, 1),
           "val": evaluate(predict(model, Xva_ids), Yva, best_thr),
           "test": evaluate(predict(model, Xte_ids), Yte, best_thr),
           "config": {"maxlen": MAXLEN, "emb": EMB, "filters": NF, "kernel": KERNEL, "vocab": V, "batch": BATCH, "lr": LR}}
json.dump(cnn_res, open(os.path.join(ART, "cnn_metrics.json"), "w"), indent=2)
json.dump(curve, open(os.path.join(ART, "cnn_train_curve.json"), "w"))
json.dump(itos, open(os.path.join(CACHE, "vocab.json"), "w"))
torch.save(best_state, os.path.join(CACHE, "cnn.pt"))
print("CNN test metrics:", cnn_res["test"])

## 5. Baseline vs deep model

Read the numbers in the table honestly. The CNN **edges the baseline on micro-F1 and precision@5**, the metrics
that track the frequent codes and the quality of the top few suggestions. The **baseline keeps a slight edge on
macro-F1**, which weights every code equally: a reminder that a well-tuned linear model is strong on the rarer
codes too, especially since we cap the CNN at the first 1,500 words. Overall the two are close on accuracy. The
deep model's decisive advantage is something the baseline cannot do at all: **explain each prediction**, the next
section.

In [ ]:
base_res = json.load(open(os.path.join(ART, "baseline_metrics.json")))
bt, ct = base_res["test"], cnn_res["test"]
print(f"{'metric':<14}{'baseline':>12}{'CNN':>12}")
for k, name in [("micro_f1", "Micro-F1"), ("macro_f1", "Macro-F1"), ("p@5", "Precision@5"), ("p@8", "Precision@8")]:
    print(f"{name:<14}{bt[k]:>12.3f}{ct[k]:>12.3f}")

metrics = [("micro_f1", "Micro-F1"), ("macro_f1", "Macro-F1"), ("p@5", "Prec@5"), ("p@8", "Prec@8")]
x = np.arange(len(metrics)); w = 0.38
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.bar(x - w/2, [bt[m] for m, _ in metrics], w, label="TF-IDF + LogReg", color=NAVY)
ax.bar(x + w/2, [ct[m] for m, _ in metrics], w, label="CNN + attention", color=TEAL)
for xi, (m, _) in enumerate(metrics):
    ax.text(xi - w/2, bt[m] + 0.008, f"{bt[m]:.3f}", ha="center", fontsize=8.5, color=SLATE)
    ax.text(xi + w/2, ct[m] + 0.008, f"{ct[m]:.3f}", ha="center", fontsize=8.5, color=SLATE)
ax.set_xticks(x); ax.set_xticklabels([n for _, n in metrics]); ax.set_ylim(0, 0.72)
ax.legend(frameon=False, fontsize=9); ax.grid(axis="y", color="#E6E9EF", lw=0.8); style(ax)
ax.set_title("Baseline vs CNN on the held-out test set", color=NAVY, fontsize=12, fontweight="bold", loc="left")
fig.tight_layout(); fig.savefig(os.path.join(RES, "fig_compare.png"), facecolor="white", bbox_inches="tight"); plt.show()

# learning curve
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(curve) + 1), curve, marker="o", color=NAVY, lw=2)
bi = int(np.argmax(curve)); ax.scatter([bi + 1], [curve[bi]], color=ACCENT, zorder=5)
ax.set_xlabel("Epoch", color=SLATE); ax.set_ylabel("Validation micro-F1", color=SLATE); style(ax)
ax.set_title("CNN learning curve", color=NAVY, fontsize=12, fontweight="bold", loc="left")
fig.tight_layout(); fig.savefig(os.path.join(RES, "fig_curve.png"), facecolor="white", bbox_inches="tight"); plt.show()

## 6. Why did it predict that? Attention as explanation

This is the payoff of per-label attention. For a single note, each predicted code exposes the phrase its attention
focused on most. The model does not just output *acute kidney failure*; it says *I read this phrase here*. A coder
can confirm or reject in seconds, and when the model is wrong the highlighted phrase usually shows you why. For
healthcare, a prediction you can interrogate is far more trustworthy than a black box with the same score.

In [ ]:
def explain(text, k=5, phrases_per=2, win=4):
    """Return the top-k predicted codes for a note, each with the phrase(s) its attention peaked on."""
    toks = text.split()[:MAXLEN]; ids = [stoi.get(w, 1) for w in toks]
    with torch.no_grad():
        logits, a = model(torch.tensor([ids]))
    probs = torch.sigmoid(logits)[0].numpy(); a = a[0].numpy()          # a: (T, C) attention per label
    out = []
    for ci in np.argsort(-probs)[:k]:
        w = a[:len(toks), ci]; chosen = []
        for p in np.argsort(-w):                                        # greedily take non-overlapping peaks
            if any(abs(p - q) < win for q in chosen): continue
            chosen.append(p)
            if len(chosen) >= phrases_per: break
        phrases = [{"text": " ".join(toks[max(0, p - win):p + win + 1]), "weight": round(float(w[p]), 3)} for p in chosen]
        out.append({"code": CODES[ci], "title": TITLE[CODES[ci]], "prob": round(float(probs[ci]), 3), "phrases": phrases})
    return out

# choose an illustrative test note: several true codes, moderate length, model recovers most of them
te_ = te.reset_index(drop=True); pick = None
for i in te_.index:
    true = {c for c in CODES if te_.loc[i, c] == 1}
    if 4 <= len(true) <= 8 and 400 <= te_.loc[i, "n_tokens"] <= 1400:
        if len({r["code"] for r in explain(te_.loc[i, "text"], k=6)} & true) >= 4:
            pick = i; break
if pick is None: pick = int(te_.n_tokens.sub(800).abs().idxmin())

note = te_.loc[pick]; true = {c for c in CODES if note[c] == 1}
evidence = explain(note.text, k=5); [r.update(is_true=r["code"] in true) for r in evidence]
json.dump({"n_tokens": int(note.n_tokens), "n_true_codes": len(true), "evidence": evidence},
          open(os.path.join(ART, "attention_examples.json"), "w"), indent=2)

for r in evidence:
    mark = "correct" if r["is_true"] else "not a coded dx"
    print(f"[{mark:>14}] p={r['prob']:.2f}  {r['title']} ({r['code']})")
    print(f"                 evidence: \"... {r['phrases'][0]['text']} ...\"")

In [ ]:
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(11, 0.9 + 1.05 * len(evidence))); ax.axis("off")
ax.set_xlim(0, 10); ax.set_ylim(0, len(evidence) + 0.6)
ax.add_patch(plt.Rectangle((0, len(evidence) + 0.15), 10, 0.5, color=NAVY, ec="none"))
ax.text(0.15, len(evidence) + 0.40, "What the model read: attention evidence for one discharge summary",
        color="white", fontsize=13, fontweight="bold", va="center")
for j, r in enumerate(evidence):
    y = len(evidence) - 1 - j + 0.15
    mark, mc = ("correct", TEAL) if r["is_true"] else ("not a coded dx", ACCENT)
    ax.add_patch(FancyBboxPatch((0.1, y), 9.8, 0.9, boxstyle="round,pad=0.02,rounding_size=0.03",
                                fc=LIGHT, ec="#D5DAE2", lw=0.8))
    ax.text(0.28, y + 0.62, f"{r['title']}  (ICD-9 {r['code']})", color=NAVY, fontsize=11.5, fontweight="bold", va="center")
    ax.text(9.7, y + 0.62, f"p={r['prob']:.2f}  -  {mark}", color=mc, fontsize=9.5, va="center", ha="right", fontweight="bold")
    ax.text(0.28, y + 0.25, f'"... {r["phrases"][0]["text"]} ..."', color=SLATE, fontsize=10, va="center",
            style="italic", fontfamily="monospace")
fig.savefig(os.path.join(RES, "attention_evidence.png"), bbox_inches="tight", pad_inches=0.1, facecolor="white"); plt.show()

## 7. Limitations and the honest read

- **Scope.** We used the 50 most frequent codes on a benchmark-sized sample. Real coding spans thousands of codes,
  including rare ones that carry the most weight, and those are much harder.
- **Truncation.** The CNN reads the first 1,500 words of each note. Discharge summaries often place the discharge
  diagnosis list near the end, so we may discard some of the most predictive text. Reading the full note is a clear
  next step.
- **Label quality.** ICD codes are billing labels entered by humans who disagree with each other. They are not
  perfect ground truth, which caps how well any model can score against them.
- **Intended use.** This is a first-pass *suggestion* tool for a human coder to confirm. It is not an autonomous
  biller and not a clinical device.
- **Generalization.** Everything here is MIMIC-III intensive-care notes from one hospital. Another hospital or ward
  would require revalidation.

## 8. Conclusion

We turned a real clinical chore, medical coding, into a well-posed machine-learning problem, built the data
carefully (splitting by patient so we never fooled ourselves), and chose metrics that respect rare codes. A
transparent baseline set a strong bar; a convolutional network with per-label attention matched it on overall
accuracy and, unlike the baseline, pointed to the exact phrase behind every code it predicted.

The lesson worth keeping: in healthcare, a model that highlights the phrases it attended to is worth more than a marginally higher
score you cannot interrogate. One note in, the right codes out, and the evidence to trust them.